In [ ]:

# 1. Import libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:

# 2. Dataset path
data_path = "fashion mnist"


In [ ]:

# 3. Load and preprocess images
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    color_mode='rgb',
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    color_mode='rgb',
    class_mode='categorical',
    subset='validation'
)


In [ ]:

# 4. Load pretrained VGG16 model
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


In [ ]:

# 5. Freeze base model layers
base_model.trainable = False


In [ ]:

# 6. Add custom classification layers
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:

# 7. Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:

# 8. Train model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)


In [ ]:

# 9. Evaluate model
loss, accuracy = model.evaluate(val_data)

print("Loss:", loss)
print("Accuracy:", accuracy)


In [ ]:

# 10. Visualize training accuracy
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.show()


In [ ]:

# 11. Fine tuning
base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=2
)


In [ ]:

# 12. Visualize prediction
images, labels = next(val_data)

prediction = model.predict(images[:1])
class_names = list(train_data.class_indices.keys())

predicted_class = class_names[np.argmax(prediction[0])]

plt.imshow(images[0])
plt.title("Predicted: " + predicted_class)
plt.axis('off')
plt.show()
